# ClaimCheck — Full Pipeline Demo
**Qwen2.5-3B-Instruct + QLoRA Adapter · Gradio UI · Google Colab**

Upload a **claim form** and a **discharge summary** (text, PDF, or image).
ClaimCheck will:
1. **Read** the documents (OCR for scans/images)
2. **Extract** 9 canonical fields using the fine-tuned SLM
3. **Compare** fields between the two documents
4. **Verdict**: flag mismatches for human review

> Woxsen University · SLM Assignment · Shaik Shazia Mahek

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────
!pip install -q \
    "transformers>=4.46.0,<5.0.0" \
    "peft>=0.13.0,<1.0.0" \
    "bitsandbytes>=0.44.0" \
    "accelerate>=1.0.0" \
    gradio \
    pytesseract \
    Pillow \
    PyMuPDF

# Install Tesseract OCR engine
!apt-get update -qq && apt-get install -y -qq tesseract-ocr > /dev/null 2>&1

import transformers, peft, torch, gradio, pytesseract
print(f"✅ All dependencies installed")
print(f"   transformers={transformers.__version__}")
print(f"   peft={peft.__version__}")
print(f"   torch={torch.__version__}")
print(f"   gradio={gradio.__version__}")
assert torch.cuda.is_available(), "❌ No GPU! Go to Runtime > Change runtime type > T4 GPU"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ Tesseract: {pytesseract.get_tesseract_version()}")

In [ ]:
# ── Step 2: Upload your LoRA adapter zip ─────────────────────────────────
import os, zipfile
from google.colab import files

ADAPTER_DIR = "./claimcheck-adapter"

if not os.path.exists(ADAPTER_DIR):
    print("📤 Upload your claimcheck-qwen2.5-3b-lora-adapter.zip file:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    print(f"Extracting {zip_name}...")
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print(f"✅ Adapter extracted to {ADAPTER_DIR}")
    for f in os.listdir(ADAPTER_DIR):
        print(f"   {f}")
else:
    print(f"✅ Adapter already exists at {ADAPTER_DIR}")
    for f in os.listdir(ADAPTER_DIR):
        print(f"   {f}")

In [ ]:
# ── Step 3: Load base model + LoRA adapter ───────────────────────────────
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
ADAPTER_DIR = "./claimcheck-adapter"

# 4-bit quantization (same as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading {MODEL_ID} + LoRA adapter...")

tokenizer = AutoTokenizer.from_pretrained(
    ADAPTER_DIR,
    trust_remote_code=True,
    padding_side="right",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
    attn_implementation="eager",
)

# Load LoRA adapter on top
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print(f"✅ Model loaded with LoRA adapter")
print(f"GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# ── Step 4: Core pipeline functions ───────────────────────────────────────
import json
import re
import hashlib
import datetime
import pytesseract
from PIL import Image
import fitz  # PyMuPDF

# ── System prompt (same as training) ──
SYSTEM_PROMPT = """You are ClaimCheck, a medical document field extractor for Indian hospital claims.

Given a hospital document (claim form or discharge summary), extract these fields with EXACT quotes from the text:
- patient_name, age, gender, hospital_name, admission_date, discharge_date, diagnosis, procedure, total_amount

For EACH field, provide:
1. "value": The extracted value (cleaned up)
2. "quote": The EXACT text span from the document where you found this

Output valid JSON only. If a field is not found, set value to null and quote to null."""

CANONICAL_FIELDS = [
    "patient_name", "age", "gender", "hospital_name",
    "admission_date", "discharge_date", "diagnosis",
    "procedure", "total_amount",
]

# ── 1. Document Reader (OCR) ──

def read_document(file_obj):
    """Read text from a file (plain text, PDF, or image).
    Returns (text, method). Never crashes: on failure returns (None, reason)."""
    import os
    if file_obj is None:
        return None, "No file uploaded"

    # Gradio may hand us a str path, a pathlib path, or an object with .name
    if isinstance(file_obj, (str, os.PathLike)):
        file_path = str(file_obj)
    else:
        file_path = str(getattr(file_obj, "name", file_obj))

    if not os.path.exists(file_path):
        return None, f"File not found on server: {file_path}"

    with open(file_path, "rb") as f:
        header = f.read(8)
    ext = os.path.splitext(file_path)[1].lower()   # [1] is the extension

    try:
        if header[:5] == b"%PDF-" or ext == ".pdf":
            return read_pdf(file_path)
        if (header[:8] == b"\x89PNG\r\n\x1a\n" or header[:2] == b"\xff\xd8"
                or ext in (".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp")):
            return read_image(file_path)
    except Exception as e:
        return None, f"Could not read {ext or 'file'}: {e}"

    # Otherwise treat as plain text
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        text = f.read()
    if text.strip():
        return text, "text"
    return None, "File is empty"

def read_pdf(file_path):
    """Extract text from PDF, falling back to OCR if needed."""
    doc = fitz.open(file_path)
    text_parts = []
    used_ocr = False
    
    for page in doc:
        # Try text extraction first
        page_text = page.get_text().strip()
        if len(page_text) > 50:
            text_parts.append(page_text)
        else:
            # Scanned page: render to image and OCR
            pix = page.get_pixmap(dpi=300)
            img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            page_text = pytesseract.image_to_string(img)
            text_parts.append(page_text)
            used_ocr = True
    
    doc.close()
    full_text = "\n\n".join(text_parts)
    method = "pdf+ocr" if used_ocr else "pdf"
    return full_text, method

def read_image(file_path):
    """OCR an image file."""
    img = Image.open(file_path)
    text = pytesseract.image_to_string(img)
    return text, "ocr"


# ── 2. SLM Field Extractor ──

def extract_fields(text, doc_type_label="Document"):
    """Run the fine-tuned model to extract fields from document text."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Extract all fields from this {doc_type_label}:\n\n{text}"}
    ]
    
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(
        input_text, return_tensors="pt"
    ).to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    
    # Parse JSON from response (with repair for common issues)
    parsed = parse_model_json(response)
    return parsed, response

def parse_model_json(text):
    """Parse JSON from model output, with repair for common issues."""
    # Direct parse
    try:
        return json.loads(text)
    except:
        pass
    
    # Find JSON block
    match = re.search(r'\{[\s\S]*\}', text)
    if match:
        json_str = match.group()
        try:
            return json.loads(json_str)
        except:
            # Try to repair common issues
            repaired = repair_json(json_str)
            try:
                return json.loads(repaired)
            except:
                pass
    return None

def repair_json(text):
    """Fix common JSON issues from the model."""
    # Fix missing closing quotes before newline
    text = re.sub(r'("quote":\s*"[^"]*?)\n', r'\1"\n', text)
    # Fix trailing comma before }
    text = re.sub(r',\s*}', '}', text)
    # Fix trailing comma before ]
    text = re.sub(r',\s*]', ']', text)
    # Fix missing quotes around field names
    text = re.sub(r'(\{|,)\s*(\w+)\s*:', r'\1"\2":', text)
    # Ensure closing brace
    if text.count('{') > text.count('}'):
        text += '}' * (text.count('{') - text.count('}'))
    return text


# ── 3. Field Comparator ──

def compare_fields(claim_fields, discharge_fields):
    """Compare extracted fields between claim form and discharge summary."""
    results = []
    
    for field in CANONICAL_FIELDS:
        claim_val = get_field_value(claim_fields, field)
        disc_val = get_field_value(discharge_fields, field)
        
        # Skip total_amount for discharge (it's claim-only)
        if field == "total_amount" and disc_val is None:
            results.append({
                "field": field,
                "claim_value": claim_val,
                "discharge_value": "N/A (claim-only field)",
                "status": "info",
                "note": "Total amount appears only on claim forms"
            })
            continue
        
        if claim_val is None and disc_val is None:
            results.append({
                "field": field,
                "claim_value": None,
                "discharge_value": None,
                "status": "missing",
                "note": "Field not found in either document"
            })
            continue
        
        if claim_val is None or disc_val is None:
            results.append({
                "field": field,
                "claim_value": claim_val,
                "discharge_value": disc_val,
                "status": "warning",
                "note": f"Field missing from {'claim form' if claim_val is None else 'discharge summary'}"
            })
            continue
        
        # Normalize for comparison
        match = values_match(claim_val, disc_val, field)
        
        results.append({
            "field": field,
            "claim_value": claim_val,
            "discharge_value": disc_val,
            "status": "match" if match else "mismatch",
            "note": "" if match else "VALUES DO NOT MATCH"
        })
    
    return results

def get_field_value(fields_dict, field_name):
    """Get a field value, handling nested format."""
    if fields_dict is None or field_name not in fields_dict:
        return None
    val = fields_dict[field_name]
    if isinstance(val, dict):
        v = val.get("value")
        if v is None or str(v).lower() in ("null", "none", ""):
            return None
        return str(v).strip()
    if val is None or str(val).lower() in ("null", "none", ""):
        return None
    return str(val).strip()

def _to_amount(s):
    s = re.sub(r"(?i)rs\.?|inr|₹|/-", "", s).replace(",", "")
    m = re.search(r"\d+(?:\.\d+)?", s)
    return float(m.group()) if m else None

def _to_date(s):
    s = s.strip()
    for fmt in ("%d-%m-%Y", "%d/%m/%Y", "%d.%m.%Y", "%Y-%m-%d", "%d-%m-%y",
                "%d/%m/%y", "%d %B %Y", "%d %b %Y", "%d-%b-%Y", "%B %d, %Y", "%b %d, %Y"):
        try:
            return datetime.datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    return None

def _words(s):
    return re.findall(r"[a-z0-9]+", s.lower())

def values_match(val1, val2, field_name):
    """Field-aware comparison. Note: plain 'x in y' is NOT safe for short fields
    (e.g. 'male' is inside 'female', '5' is inside '52')."""
    v1, v2 = val1.lower().strip(), val2.lower().strip()
    if v1 == v2:
        return True

    if field_name == "total_amount":
        a, b = _to_amount(v1), _to_amount(v2)
        return a is not None and a == b

    if field_name == "age":
        a, b = re.search(r"\d+", v1), re.search(r"\d+", v2)
        return bool(a and b and a.group() == b.group())

    if field_name == "gender":
        g = lambda s: "f" if s.startswith("f") else ("m" if s.startswith("m") else s)
        return g(v1) == g(v2)

    if field_name in ("admission_date", "discharge_date"):
        d1, d2 = _to_date(v1), _to_date(v2)
        if d1 and d2:
            return d1 == d2
        return re.sub(r"[/\-.\s]", "-", v1) == re.sub(r"[/\-.\s]", "-", v2)

    w1, w2 = _words(v1), _words(v2)
    if field_name == "patient_name":
        titles = {"mr", "mrs", "ms", "smt", "shri", "dr", "miss"}
        return {w for w in w1 if w not in titles} == {w for w in w2 if w not in titles}

    # hospital_name, diagnosis, procedure: same words, or one is a whole-word
    # subset of the other (handles partial extraction)
    if w1 == w2:
        return True
    s1, s2 = set(w1), set(w2)
    return bool(s1 and s2) and (s1 <= s2 or s2 <= s1)


# ── 4. Verdict Generator ──

def generate_verdict(comparison_results):
    """Generate overall verdict from comparison results."""
    mismatches = [r for r in comparison_results if r["status"] == "mismatch"]
    warnings = [r for r in comparison_results if r["status"] == "warning"]
    matches = [r for r in comparison_results if r["status"] == "match"]
    
    if len(mismatches) == 0 and len(warnings) == 0:
        verdict = "APPROVED"
        confidence = "HIGH"
        summary = f"All {len(matches)} comparable fields match between claim form and discharge summary."
    elif len(mismatches) == 0 and len(warnings) > 0:
        verdict = "REVIEW RECOMMENDED"
        confidence = "MEDIUM"
        summary = f"{len(matches)} fields match, but {len(warnings)} field(s) could not be compared (missing from one document)."
    elif len(mismatches) <= 2:
        verdict = "FLAGGED FOR REVIEW"
        confidence = "LOW"
        mismatch_fields = ", ".join(r["field"] for r in mismatches)
        summary = f"Found {len(mismatches)} mismatch(es) in: {mismatch_fields}. Manual review required."
    else:
        verdict = "REJECTED"
        confidence = "VERY LOW"
        mismatch_fields = ", ".join(r["field"] for r in mismatches)
        summary = f"Found {len(mismatches)} mismatches in: {mismatch_fields}. Likely fraudulent or incorrect claim."
    
    return {
        "verdict": verdict,
        "confidence": confidence,
        "summary": summary,
        "matches": len(matches),
        "mismatches": len(mismatches),
        "warnings": len(warnings),
    }


# ── 5. Audit Log ──

audit_log = []

def add_audit_entry(claim_file, discharge_file, comparison, verdict_info):
    """Add a hash-chained audit log entry."""
    prev_hash = audit_log[-1]["hash"] if audit_log else "0" * 64
    
    entry = {
        "id": len(audit_log) + 1,
        "timestamp": datetime.datetime.now().isoformat(),
        "claim_file": os.path.basename(claim_file) if claim_file else "N/A",
        "discharge_file": os.path.basename(discharge_file) if discharge_file else "N/A",
        "verdict": verdict_info["verdict"],
        "confidence": verdict_info["confidence"],
        "mismatches": verdict_info["mismatches"],
        "prev_hash": prev_hash,
    }
    
    # Hash-chain: hash of current entry + previous hash
    entry_str = json.dumps(entry, sort_keys=True)
    entry["hash"] = hashlib.sha256((entry_str + prev_hash).encode()).hexdigest()
    
    audit_log.append(entry)
    return entry

print("✅ Pipeline functions loaded:")
print("   - read_document()   : OCR / text extraction")
print("   - extract_fields()  : SLM field extraction")
print("   - compare_fields()  : field-by-field comparison")
print("   - generate_verdict(): approval / rejection")
print("   - add_audit_entry() : hash-chained audit log")

In [ ]:
# ── Step 5: Launch Gradio Interface ───────────────────────────────────────
import gradio as gr
import os

def _process_claim(claim_file, discharge_file):
    """Main pipeline: read docs -> extract -> compare -> verdict."""
    
    if claim_file is None or discharge_file is None:
        return "Please upload both files.", "", "", "", ""
    
    status_parts = []
    
    # ── Step 1: Read documents ──
    status_parts.append("Step 1/4: Reading documents...")
    
    claim_text, claim_method = read_document(claim_file)
    discharge_text, disc_method = read_document(discharge_file)
    
    if claim_text is None:
        return "Could not read claim form file.", "", "", "", ""
    if discharge_text is None:
        return "Could not read discharge summary file.", "", "", "", ""
    
    status_parts.append(f"  Claim form: read via {claim_method} ({len(claim_text)} chars)")
    status_parts.append(f"  Discharge summary: read via {disc_method} ({len(discharge_text)} chars)")
    
    # ── Step 2: Extract fields ──
    status_parts.append("\nStep 2/4: Extracting fields with SLM...")
    
    claim_fields, claim_raw = extract_fields(claim_text, "Claim Form")
    discharge_fields, disc_raw = extract_fields(discharge_text, "Discharge Summary")
    
    if claim_fields is None:
        status_parts.append("  ⚠️  Could not parse claim form extraction (JSON error)")
        status_parts.append(f"  Raw output: {claim_raw[:300]}")
    else:
        populated = sum(1 for f in CANONICAL_FIELDS if get_field_value(claim_fields, f) is not None)
        status_parts.append(f"  Claim form: {populated}/{len(CANONICAL_FIELDS)} fields extracted")
    
    if discharge_fields is None:
        status_parts.append("  ⚠️  Could not parse discharge summary extraction (JSON error)")
        status_parts.append(f"  Raw output: {disc_raw[:300]}")
    else:
        populated = sum(1 for f in CANONICAL_FIELDS if get_field_value(discharge_fields, f) is not None)
        status_parts.append(f"  Discharge summary: {populated}/{len(CANONICAL_FIELDS)} fields extracted")
    
    # ── Step 3: Compare ──
    status_parts.append("\nStep 3/4: Comparing fields...")
    
    if claim_fields and discharge_fields:
        comparison = compare_fields(claim_fields, discharge_fields)
    else:
        comparison = []
        status_parts.append("  Skipped (extraction failed for one or both documents)")
    
    # ── Step 4: Verdict ──
    status_parts.append("\nStep 4/4: Generating verdict...")
    
    if comparison:
        verdict_info = generate_verdict(comparison)
        audit_entry = add_audit_entry(claim_file, discharge_file, comparison, verdict_info)
    else:
        verdict_info = {
            "verdict": "CANNOT DETERMINE",
            "confidence": "NONE",
            "summary": "Could not extract fields from one or both documents.",
            "matches": 0, "mismatches": 0, "warnings": 0,
        }
    
    status_parts.append("\n✅ Pipeline complete!")
    
    # ── Format outputs ──
    # 1. Status log
    status_text = "\n".join(status_parts)
    
    # 2. Extracted fields (side by side)
    fields_md = format_fields_table(claim_fields, discharge_fields)
    
    # 3. Comparison table
    comparison_md = format_comparison_table(comparison)
    
    # 4. Verdict
    verdict_md = format_verdict(verdict_info)
    
    # 5. Audit log
    audit_md = format_audit_log()
    
    return status_text, fields_md, comparison_md, verdict_md, audit_md

def format_fields_table(claim_fields, discharge_fields):
    """Format extracted fields as markdown table."""
    rows = ["| Field | Claim Form | Discharge Summary |",
            "|-------|-----------|-------------------|"]
    for field in CANONICAL_FIELDS:
        cv = get_field_value(claim_fields, field) or "-"
        dv = get_field_value(discharge_fields, field) or "-"
        rows.append(f"| **{field}** | {cv} | {dv} |")
    return "\n".join(rows)

def format_comparison_table(comparison):
    """Format comparison results as markdown."""
    if not comparison:
        return "No comparison available."
    
    rows = ["| Field | Status | Claim Value | Discharge Value | Note |",
            "|-------|--------|-------------|-----------------|------|"]
    
    status_emoji = {
        "match": "✅", "mismatch": "❌", "warning": "⚠️",
        "missing": "➖", "info": "ℹ️"
    }
    
    for r in comparison:
        emoji = status_emoji.get(r["status"], "")
        cv = r["claim_value"] or "-"
        dv = r["discharge_value"] or "-"
        rows.append(f"| **{r['field']}** | {emoji} {r['status'].upper()} | {cv} | {dv} | {r['note']} |")
    return "\n".join(rows)

def format_verdict(verdict_info):
    """Format verdict as styled markdown."""
    emoji_map = {
        "APPROVED": "✅",
        "REVIEW RECOMMENDED": "⚠️",
        "FLAGGED FOR REVIEW": "🔍",
        "REJECTED": "❌",
        "CANNOT DETERMINE": "❓",
    }
    emoji = emoji_map.get(verdict_info["verdict"], "")
    
    lines = [
        f"## {emoji} {verdict_info['verdict']}",
        f"**Confidence:** {verdict_info['confidence']}",
        f"",
        f"{verdict_info['summary']}",
        f"",
        f"| Matches | Mismatches | Warnings |",
        f"|---------|------------|----------|",
        f"| {verdict_info['matches']} | {verdict_info['mismatches']} | {verdict_info['warnings']} |",
    ]
    return "\n".join(lines)

def format_audit_log():
    """Format audit log as markdown table."""
    if not audit_log:
        return "No entries yet."
    
    rows = ["| # | Timestamp | Claim | Discharge | Verdict | Hash (first 16) |",
            "|---|-----------|-------|-----------|---------|-----------------|"]
    for entry in audit_log:
        ts = entry["timestamp"][:19]
        rows.append(
            f"| {entry['id']} | {ts} | {entry['claim_file'][:20]} | "
            f"{entry['discharge_file'][:20]} | {entry['verdict']} | "
            f"`{entry['hash'][:16]}...` |"
        )
    return "\n".join(rows)


# ── Process plain text input ──
def _process_text_input(claim_text, discharge_text):
    """Process when user pastes text directly instead of uploading files."""
    if not claim_text.strip() or not discharge_text.strip():
        return "Please enter text for both documents.", "", "", "", ""
    
    status_parts = []
    
    # ── Extract fields ──
    status_parts.append("Step 1/3: Extracting fields from claim form...")
    claim_fields, claim_raw = extract_fields(claim_text, "Claim Form")
    
    status_parts.append("Step 2/3: Extracting fields from discharge summary...")
    discharge_fields, disc_raw = extract_fields(discharge_text, "Discharge Summary")
    
    if claim_fields is None:
        status_parts.append(f"  ⚠️ Claim form extraction failed")
    else:
        populated = sum(1 for f in CANONICAL_FIELDS if get_field_value(claim_fields, f) is not None)
        status_parts.append(f"  Claim form: {populated} fields extracted")
    
    if discharge_fields is None:
        status_parts.append(f"  ⚠️ Discharge summary extraction failed")
    else:
        populated = sum(1 for f in CANONICAL_FIELDS if get_field_value(discharge_fields, f) is not None)
        status_parts.append(f"  Discharge summary: {populated} fields extracted")
    
    # ── Compare ──
    status_parts.append("\nStep 3/3: Comparing and generating verdict...")
    if claim_fields and discharge_fields:
        comparison = compare_fields(claim_fields, discharge_fields)
        verdict_info = generate_verdict(comparison)
        add_audit_entry("text-input", "text-input", comparison, verdict_info)
    else:
        comparison = []
        verdict_info = {"verdict": "CANNOT DETERMINE", "confidence": "NONE",
                        "summary": "Extraction failed.", "matches": 0,
                        "mismatches": 0, "warnings": 0}
    
    status_parts.append("\n✅ Done!")
    
    return ("\n".join(status_parts),
            format_fields_table(claim_fields, discharge_fields),
            format_comparison_table(comparison),
            format_verdict(verdict_info),
            format_audit_log())


import traceback

def process_claim(claim_file, discharge_file):
    try:
        return _process_claim(claim_file, discharge_file)
    except Exception:
        tb = traceback.format_exc()
        print(tb)
        return "ERROR:\n" + tb, "", "", "## ❌ Pipeline error (see Processing Log)", ""

def process_text_input(claim_text, discharge_text):
    try:
        return _process_text_input(claim_text, discharge_text)
    except Exception:
        tb = traceback.format_exc()
        print(tb)
        return "ERROR:\n" + tb, "", "", "## ❌ Pipeline error (see Processing Log)", ""


# ── Build Gradio UI ──
with gr.Blocks(
    title="ClaimCheck",
    theme=gr.themes.Soft(primary_hue="blue"),
) as demo:
    gr.Markdown("""
# 🏥 ClaimCheck
### AI-Powered Health Insurance Claim Verification
Upload a **claim form** and a **discharge summary** to verify consistency.  
Powered by **Qwen2.5-3B-Instruct** fine-tuned with QLoRA on Indian hospital documents.
    """)
    
    with gr.Tabs():
        # ── Tab 1: File Upload ──
        with gr.TabItem("📁 File Upload"):
            with gr.Row():
                claim_file = gr.File(label="Claim Form (text / PDF / image)", type="filepath")
                discharge_file = gr.File(label="Discharge Summary (text / PDF / image)", type="filepath")
            
            file_btn = gr.Button("🔍 Verify Claim", variant="primary", size="lg")
        
        # ── Tab 2: Paste Text ──
        with gr.TabItem("📝 Paste Text"):
            with gr.Row():
                claim_text_input = gr.Textbox(
                    label="Claim Form Text", lines=12,
                    placeholder="Paste claim form text here..."
                )
                discharge_text_input = gr.Textbox(
                    label="Discharge Summary Text", lines=12,
                    placeholder="Paste discharge summary text here..."
                )
            
            text_btn = gr.Button("🔍 Verify Claim", variant="primary", size="lg")
    
    # ── Output sections ──
    with gr.Accordion("📊 Processing Log", open=True):
        status_output = gr.Textbox(label="Status", lines=10, interactive=False)
    
    verdict_output = gr.Markdown(label="Verdict")
    
    with gr.Accordion("📋 Extracted Fields", open=True):
        fields_output = gr.Markdown()
    
    with gr.Accordion("🔍 Field-by-Field Comparison", open=True):
        comparison_output = gr.Markdown()
    
    with gr.Accordion("📜 Audit Log", open=False):
        audit_output = gr.Markdown()
    
    # ── Wire up buttons ──
    outputs = [status_output, fields_output, comparison_output, verdict_output, audit_output]
    
    file_btn.click(
        fn=process_claim,
        inputs=[claim_file, discharge_file],
        outputs=outputs,
    )
    
    text_btn.click(
        fn=process_text_input,
        inputs=[claim_text_input, discharge_text_input],
        outputs=outputs,
    )

print("🚀 Launching ClaimCheck...")
demo.launch(share=True, debug=True)